In [1]:
import json
from datetime import datetime as dt
from pathlib import Path

import polars as pl
import yaml
from dateutil.relativedelta import relativedelta

from clean import clean_os_version, merge_partitions
from common import daterange, read_day

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))


In [2]:
def collect_os_stats(days: list) -> pl.DataFrame:
    os_stats = pl.DataFrame()
    for i in days:
        data = (
            read_day(i, config["dirs"]["raw_source"])
            .with_columns(
                pl.concat_str(
                    [pl.col("device_aid_hashed"), pl.col("device_aid_type")],
                    separator=".",
                ).alias("id_with_type"),
                pl.from_epoch(pl.col("timestamp"), time_unit="s")
                .dt.convert_time_zone(time_zone="Europe/Budapest")
                .dt.date()
                .alias("day"),
            )
            .rename({"device_aid_hashed": "device_id"})
        )
        data = data.select(
            pl.col.device_id,
            pl.col.id_with_type,
            pl.col.OS,
            pl.col.OS_version,
            pl.col.day,
        ).unique()
        data = data.with_columns(
            pl.col.OS_version.str.extract(r"(\d+)\..*", 1).cast(pl.Int16),
            pl.col("id_with_type").replace_strict(lookup),
        )
        os_stats = pl.concat([os_stats, data])
    return os_stats

In [3]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}"

In [4]:
Path(f"{OUTPUT}/descriptives").mkdir(parents=True, exist_ok=True)

In [5]:
with open(f"{OUTPUT}/id_lookup/id_lookup.json", "r") as fp:
    lookup = json.load(fp)

In [6]:
try:
    os_stats = pl.read_parquet(f"{OUTPUT}/descriptives/os_stats.parquet")
except FileNotFoundError:
    # runs 50 m
    os_stats = collect_os_stats()
    os_stats.columns = ["id_with_type", "device_id", "OS", "OS_version", "day"]
    os_stats = os_stats.sort(["device_id", "day"])
    os_stats.write_parquet(f"{OUTPUT}/descriptives/os_stats.parquet")

In [7]:
os_type_count = os_stats.group_by(["device_id", "OS"]).agg(
    pl.col.id_with_type.unique().count().alias("count")
)

In [8]:
os_type_count["count"].sum() == len(os_type_count)

True

In [9]:
base = (
    os_stats.drop("id_with_type")
    .sort(
        ["device_id", "OS", "OS_version", "day"], descending=[False, False, False, True]
    )
    .unique(["device_id", "OS"], maintain_order=True, keep="first")
    .filter(pl.col.OS_version.is_not_null())
)

In [ ]:
def relative_age(
    year: int, month: int, ref_year: int = 2025, ref_month: int = 12
) -> int:
    """
    >>> relative_age(2017, 4, 2017, 4)
    0
    >>> relative_age(2017, 3, 2017, 4)
    1
    >>> relative_age(2016, 4, 2017, 4)
    12
    """
    years = (
        relativedelta(dt.date(ref_year, ref_month, 1), dt.date(year, month, 1)).years
        * 12
    )
    months = relativedelta(
        dt.date(ref_year, ref_month, 1), dt.date(year, month, 1)
    ).months
    return years + months


def relative_age_from_date(day: dt.date, ref: dt.date) -> int:
    return relative_age(day.year, day.month, ref_year=ref.year, ref_month=ref.month)


def get_release_date(os_versions: dict, os: str, os_version: int) -> dt.date:
    return os_versions[os.lower()][os_version]


def get_release_date_e(os_versions: dict, os: str, os_version: int) -> dt.date:
    return os_versions[os.str.to_lowercase()][os_version]


In [27]:
with open("../data/os_version.yaml", "r") as fp:
    os_versions = yaml.safe_load(fp)

In [28]:
get_release_date(os_versions, "Android", 5)

datetime.date(2014, 11, 4)

In [55]:
base.with_columns(pl.struct(["OS", "OS_version"]).alias("a"))

device_id,OS,OS_version,day,a
i64,str,i16,date,struct[2]
8,"""Android""",5,2025-10-24,"{""Android"",5}"
30,"""Android""",10,2025-12-26,"{""Android"",10}"
32,"""iOS""",14,2025-07-18,"{""iOS"",14}"
64,"""Android""",11,2024-12-24,"{""Android"",11}"
76,"""Android""",8,2025-10-24,"{""Android"",8}"
…,…,…,…,…
1652664,"""Android""",8,2025-07-30,"{""Android"",8}"
1652673,"""Android""",8,2025-10-22,"{""Android"",8}"
1652677,"""Android""",7,2025-08-20,"{""Android"",7}"


In [56]:
# base.with_columns(
#     get_release_date(os_versions, pl.col.OS, pl.col.OS_version).alias("release_date")
# )

In [68]:
lookup = pl.DataFrame(
    [
        {"OS": os_name, "OS_version": version, "release_date": date}
        for os_name, versions in os_versions.items()
        for version, date in versions.items()
    ]
).with_columns(pl.col.OS.replace({"ios": "iOS", "android": "Android"}))

In [ ]:
enriched = (
    base.join(lookup, on=["OS", "OS_version"], how="left")
    .with_columns(
        (pl.col.day - pl.col.release_date).dt.total_days().alias("relative_age")
    )
    .rename({"day": "last_active_day"})
)

In [79]:
enriched.write_csv(f"{OUTPUT}/descriptives/phone_relative_age.csv")

# old

In [ ]:
import numpy as np


def clean_os_version_(x: str) -> float:
    result = x
    if "." in x:
        result = x.strip().split(".")[0]
    if x.strip() in ["ANDROID", "IOS", "Other", "null", ""]:
        result = np.nan
    if not result.isnumeric():
        result = np.nan
    return float(result)


device_id,OS,OS_version,contains,matches
str,str,str,bool,str
"""84b1596a84ff86e1b0dd2d3117070c…","""iOS""",null,null,null
"""d041d469ef0d1e65bb57fbadb62d57…","""Android""",null,null,null
"""b3d73e9ea9b2d5dfb9427b442477a8…","""Android""","""10.0.0""",true,"""10"""
"""5cb23f5d21463a428504ee628438b6…","""Android""",null,null,null
"""0466e10585adfd41cc60abc4c9da74…","""Android""","""10.0.0""",true,"""10"""
…,…,…,…,…
"""0ca61b735044c45f7cecbb30b5af7e…","""Android""","""11""",true,"""1"""
"""e97be53b4556c954d4e341c841f15d…","""Android""",null,null,null
"""190d5b3b7c7cf6e1c8c1de8776d27b…","""Android""","""9.0.0""",true,"""9"""


In [8]:
p00 = pl.read_parquet(f"{OUTPUT}/partitioned/00/pings.parquet")

In [14]:
p00

timestamp,device_id,device_aid_type,latitude,longitude
"datetime[μs, Europe/Budapest]",i64,str,f64,f64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154
2024-10-02 03:27:50 CEST,1,"""AAID""",47.47999,19.07423
2024-10-02 04:06:48 CEST,1,"""AAID""",47.476752,19.075154
…,…,…,…,…
2025-10-24 15:16:40 CEST,6585,"""AAID""",47.588495,19.008983
2025-10-24 15:16:55 CEST,6585,"""AAID""",47.58826,19.009709
2025-10-24 15:17:04 CEST,6585,"""AAID""",47.588517,19.009056


In [ ]:
p00.with_columns(pl.col("timestamp").dt.date()).unique(
    ["timestamp", "device_id", "device_aid_type"]
).filter(pl.col.device_id == 5710).sort("timestamp").write_csv(
    "../output/debug/5710_.csv"
)

In [9]:
p00.filter(pl.col.device_id == 5710).unique(["device_id", "device_aid_type"])

timestamp,device_id,device_aid_type,latitude,longitude
"datetime[μs, Europe/Budapest]",i64,str,f64,f64
2023-10-01 03:32:02 CEST,5710,"""IDFA""",47.522914,18.833683
2024-12-23 18:17:10 CET,5710,"""AAID""",47.483703,19.024008


In [ ]:
apple, android = p00.filter(pl.col.device_id == 5710).partition_by("device_aid_type")

In [63]:
apple

timestamp,device_id,device_aid_type,latitude,longitude
"datetime[μs, Europe/Budapest]",i64,str,f64,f64
2023-10-01 03:32:02 CEST,5710,"""IDFA""",47.522914,18.833683
2023-10-01 03:32:08 CEST,5710,"""IDFA""",47.522924,18.833674
2023-10-01 07:21:40 CEST,5710,"""IDFA""",47.522914,18.833683
2023-10-01 07:22:29 CEST,5710,"""IDFA""",47.522924,18.833674
2023-10-01 09:40:37 CEST,5710,"""IDFA""",47.522914,18.833683
…,…,…,…,…
2025-09-20 21:43:52 CEST,5710,"""IDFA""",47.528087,19.033525
2025-09-20 21:53:15 CEST,5710,"""IDFA""",47.52524,18.980963
2025-09-20 21:53:51 CEST,5710,"""IDFA""",47.525244,18.980937


In [65]:
import geopandas as gpd

In [75]:
gpd.GeoDataFrame(
    android.to_pandas(),
    geometry=gpd.points_from_xy(android["longitude"], android["latitude"]),
    crs=4326,
).to_file("../output/debug/5710_android.geojson")

In [76]:
gpd.GeoDataFrame(
    apple.to_pandas(),
    geometry=gpd.points_from_xy(apple["longitude"], apple["latitude"]),
    crs=4326,
).to_file("../output/debug/5710_apple.geojson")